In [ ]:
import galsim
import numpy
from astropy import table
from functions_images import build_psf
from matplotlib import pyplot
from pandas import DataFrame
from utils import search_folder, write_non_overlapped_with_radius

DIR_DATASETS = search_folder(name="DATASETS")
REAL_GALAXIES = DIR_DATASETS / "cosmos"

DIRS = {"images": DIR_DATASETS / "images" / "cosmos"}
DIRS["images"].mkdir(parents=True, exist_ok=True)

PIXEL_SCALE = 0.1
IM_SIZE = 64
gsparams = galsim.GSParams(
    xvalue_accuracy=2.0e-4, kvalue_accuracy=2.0e-4, maxk_threshold=5.0e-3, folding_threshold=1.0e-2
)

In [ ]:
sample = "23.5"
# sample = "25.2"


def double_filters(sample, dir):
    filtered_cosmos_cat = galsim.COSMOSCatalog(
        sample=sample,
        dir=dir,
        use_real=True,
        exclusion_level="marginal",  # 'none', 'bad_stamp', 'bad_fits' marginal
        min_hlr=0.0,
        max_hlr=0.0,
        min_flux=0,
        max_flux=0,
    )

    filtered_indices = filtered_cosmos_cat.orig_index
    subset_param_cat = filtered_cosmos_cat.param_cat[filtered_indices]

    viable_sersic_mask = subset_param_cat["viable_sersic"] != 0
    fit_status_mask = subset_param_cat["fit_status"][:, 4] == 1

    final_mask = viable_sersic_mask & fit_status_mask

    filtered_indices = filtered_indices[final_mask]
    return filtered_indices


filtered_indices = double_filters(sample, REAL_GALAXIES)
print(len(filtered_indices))

In [ ]:
all_cosmos_cat = galsim.COSMOSCatalog(
    sample=sample, dir=REAL_GALAXIES, use_real=True, exclusion_level="none", exptime=3 * 565
)
print(all_cosmos_cat.nobjects)

In [ ]:
index = 0
deep = False  # Modify fluxes and sizes of galaxies from the F814W<23.5 sample in order to roughly simulate an F814W<25 sample

index = filtered_indices[index]
real_gal = all_cosmos_cat.makeGalaxy(
    index=index,
    gal_type="real",  # 'real', parametric
    noise_pad_size=IM_SIZE * PIXEL_SCALE,  # size of region to pad with noise in arcsec.
    deep=deep,
    sersic_prec=0.0,
    gsparams=gsparams,
)
param_gal = all_cosmos_cat.makeGalaxy(
    index=index,
    gal_type="parametric",  # 'real', parametric
    noise_pad_size=IM_SIZE * PIXEL_SCALE,  # size of region to pad with noise in arcsec.
    deep=deep,
    sersic_prec=0.0,
    gsparams=gsparams,
)

In [ ]:
psf = build_psf("airy")

real_gal = galsim.Convolve(real_gal, psf)
im_real = galsim.Image(IM_SIZE, IM_SIZE)
real_gal.drawImage(image=im_real, scale=PIXEL_SCALE)

param_gal = galsim.Convolve(param_gal, psf)
im_param = galsim.Image(IM_SIZE, IM_SIZE)
param_gal.drawImage(image=im_param, scale=PIXEL_SCALE)

fig, axs = pyplot.subplots(1, 2, figsize=(8, 4))

im0 = axs[0].imshow(im_real.array, origin="lower")
axs[0].set_title(f"Real Galaxy {index}")
fig.colorbar(im0, ax=axs[0], orientation="vertical", fraction=0.05, pad=0.05)
im1 = axs[1].imshow(im_param.array, origin="lower")
axs[1].set_title(f"Parametric Model {index}")
fig.colorbar(im1, ax=axs[1], orientation="vertical", fraction=0.05, pad=0.05)

fig.tight_layout()
fig.show()

In [ ]:
param_records = all_cosmos_cat.getParametricRecord(index=index)

In [ ]:
all_param_records = list(map(all_cosmos_cat.getParametricRecord, filtered_indices))
# all_param_records[index]

In [ ]:
all_param_records = [
    (d["mag_auto"], d["flux_radius"], d["zphot"], d["hlr"], d["flux"][0], d["sersicfit"][0])
    for d in all_param_records
]
df_all_param_records = DataFrame(
    all_param_records, columns=["mag_auto", "flux_radius", "zphot", "hlr", "flux_0", "sersicfit_0"]
)

In [ ]:
columns_to_keep = ["mag_auto", "flux_radius", "sersicfit", "hlr", "flux", "zphot"]
filtered_params = {key: param_records[key] for key in columns_to_keep if key in param_records}
print(filtered_params)
# print(cat23_param[index])

In [ ]:
# TROUVER HLR + verif avec deep change les parameters!!!!

In [ ]:
cat23 = table.Table.read(REAL_GALAXIES / "real_galaxy_catalog_23.5.fits")
cat25 = table.Table.read(REAL_GALAXIES / "real_galaxy_catalog_25.2.fits")
cat23_param = table.Table.read(REAL_GALAXIES / "real_galaxy_catalog_23.5_fits.fits")
cat25_param = table.Table.read(REAL_GALAXIES / "real_galaxy_catalog_25.2_fits.fits")

print(len(cat23), len(cat23_param), len(cat25), len(cat25_param))
print(cat23.colnames)
print(cat23_param.colnames)

if sample == "23.5":
    print(cat23[index])
    catalog = cat23_param
elif sample == "25.2":
    print(cat25[index])
    catalog = cat25_param

In [ ]:
filtered_cat = catalog[catalog["viable_sersic"] != 0]
filtered_cat = filtered_cat[filtered_cat["fit_status"][:, 4] == 1]

hypercube = {
    "redshift": numpy.array(filtered_cat["zphot"]).astype(numpy.float64),
    "mag": numpy.array(filtered_cat["mag_auto"]).astype(numpy.float64),
    "n_sersic": numpy.array(filtered_cat["sersicfit"][:, 2]).astype(numpy.float64),
    "hlr": numpy.array(filtered_cat["sersicfit"][:, 1]).astype(numpy.float64),  # Pixels
}
hypercube = DataFrame(hypercube)
print(hypercube.shape[0])

In [ ]:
# flux_radius: half-light radius for the PSF-convolved object - Pixels
# flux: Model-based brightness Flux units PSF-deconvolved (intrinsic) - electrons or ADU
# hlr: Model fit Sérsic (PSF-deconvolved?) - arcsec
# MAG_AUTO: estimate of the total magnitude of an object  in real image. includes PSF.

In [ ]:
def histos(hypercube, dir_plots, bins=50):
    font_style = {"fontsize": 10, "fontweight": "bold", "family": "serif", "color": "darkgreen"}
    col_to_log = ["sersic_hlr", "sersic_flux", "flux_radius"]

    fig, axs = pyplot.subplots(1, 4, figsize=(12, 3))
    axs = axs.ravel()

    for i, (key, values) in enumerate(hypercube.items()):
        color = "skyblue" if i % 2 == 0 else "orange"
        axs[i].hist(values, bins=bins, alpha=0.7, color=color)
        axs[i].set_title(key, fontdict=font_style)
        axs[i].set_xlabel(key)
        axs[i].set_ylabel("Count")
        if key in col_to_log:
            axs[i].set_yscale("log")

    fig.tight_layout()
    fig.savefig(f"{dir_plots}.pdf", dpi=300)


histos(hypercube, DIRS["images"] / "histos", bins=50)

In [ ]:
# AJOUTER LES NON OVERLAP DE 23.2 a 25.2§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
write_non_overlapped_with_radius(REAL_GALAXIES, cat23, cat25, cat23_param, cat25_param, radius=0.1)

In [ ]:
cat23_non_overlap = table.Table.read(REAL_GALAXIES / "real_galaxy_catalog_23.5_non_overlap.fits")
cat25_non_overlap = table.Table.read(REAL_GALAXIES / "real_galaxy_catalog_25.2_non_overlap.fits")
cat23_param_non_overlap = table.Table.read(
    REAL_GALAXIES / "real_galaxy_catalog_23.5_fits_non_overlap.fits"
)
cat25_param_non_overlap = table.Table.read(
    REAL_GALAXIES / "real_galaxy_catalog_25.2_fits_non_overlap.fits"
)
print(
    len(cat23_non_overlap),
    len(cat23_param_non_overlap),
    len(cat25_non_overlap),
    len(cat25_param_non_overlap),
)
print(cat23_non_overlap.colnames)
print(cat23_param_non_overlap.colnames)